# Complete BaBar isobar CP closure: $B^\pm\to K^\pm\pi^\mp\pi^\pm$

This notebook implements the complete nominal BaBar signal model of Phys. Rev. D 78, 012004 (2008), arXiv:0803.4451, using the Cartesian CP convention
\[c_j^+=(x_j+\Delta x_j)+i(y_j+\Delta y_j),\qquad c_j^-=(x_j-\Delta x_j)+i(y_j-\Delta y_j).\]

The CP fit is performed in the **joint space of Dalitz coordinates and charge**. The fitted PDF is
\[p(\Phi,q)=\frac{|A_q(\Phi)|^2}{I_+ + I_-},\qquad I_\pm=\int |A_\pm(\Phi)|^2\,d\Phi,\]
so the relative $B^+/B^-$ yield carries information and the charge asymmetries are coupled through the common normalization.

Particle ordering is fixed to `(1,2,3)=(K^\pm,\pi^\pm,\pi^\mp)`, hence
\[s_{13}=m^2(K^\pm\pi^\mp),\qquad s_{23}=m^2(\pi^+\pi^-).\]

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, CPRealImag, DecayChannel, DecayModel, LASS, Minimizer,
    NonResonant, Parameter, Resonance, enable_x64, weighted_resample,
)
from dalitzplotfitter.likelihood import CPJointNLL

enable_x64()

## 1. BaBar Table-I CP coefficients

In [ ]:
TABLE_I = {
    # name: (x, y, dx, dy, fixed_xy, fixed_cp)
    'Kstar892':    ( 1.000,  0.000, -0.017, -0.238, True,  False),
    'KpiS':        ( 1.718, -0.727, -0.154, -0.285, False, False),
    'rho770':      ( 0.683, -0.025, -0.160,  0.169, False, False),
    'f0_980':      (-0.220,  1.203, -0.109,  0.047, False, False),
    'chic0':       (-0.263,  0.180, -0.033, -0.007, False, False),
    'NR':          (-0.594,  0.068,  0.000,  0.000, False, True),
    'K2star1430':  (-0.301,  0.424,  0.032,  0.007, False, False),
    'omega782':    (-0.058,  0.100,  0.000,  0.000, False, True),
    'f2_1270':     (-0.193,  0.110, -0.089,  0.125, False, False),
    'fX1300':      (-0.290, -0.136,  0.024,  0.056, False, False),
}

def cp_coefficient(name):
    x, y, dx, dy, fixed_xy, fixed_cp = TABLE_I[name]
    def p(suffix, value, fixed=False, bound=3.0, step=0.02):
        return Parameter.coefficient(
            f'{name}.{suffix}', value, owner=name, fixed=fixed,
            bounds=(-bound, bound), step=step,
        )
    return CPRealImag(
        p('x', x, fixed_xy), p('y', y, fixed_xy),
        p('dx', dx, fixed_cp, 1.5, 0.01), p('dy', dy, fixed_cp, 1.5, 0.01),
    )

coeff = {name: cp_coefficient(name) for name in TABLE_I}

## 2. Complete nominal dynamical model

The ten coherent terms are $K^*(892)^0$, LASS $K\pi$ S-wave, $K_2^*(1430)^0$, $\rho(770)^0$, $\omega(782)$, $f_0(980)$, $f_2(1270)$, $f_X(1300)$, $\chi_{c0}$ and a constant nonresonant amplitude.

In [ ]:
channel_plus  = DecayChannel('B+', ('K+', 'pi+', 'pi-'))
channel_minus = DecayChannel('B-', ('K-', 'pi-', 'pi+'))

def build_model(channel, charge):
    c = lambda name: coeff[name].for_charge(charge)
    R = 4.0
    return DecayModel(channel, [
        Resonance('Kstar892',   (0,2), c('Kstar892'),   mass=0.8958, width=0.0474, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('KpiS',       (0,2), c('KpiS'),       lineshape=LASS(2.07,3.32,1.8), mass=1.425, width=0.270, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('rho770',     (1,2), c('rho770'),     mass=0.7753, width=0.1491, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f0_980',     (1,2), c('f0_980'),     lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('chic0',      (1,2), c('chic0'),      mass=3.4147, width=0.0105, spin=0, resonance_radius=R, parent_radius=R),
        NonResonant(c('NR'), name='NR'),
        Resonance('K2star1430', (0,2), c('K2star1430'), mass=1.4324, width=0.109, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('omega782',   (1,2), c('omega782'),   mass=0.78265, width=0.00849, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f2_1270',    (1,2), c('f2_1270'),    mass=1.2755, width=0.1867, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('fX1300',     (1,2), c('fX1300'),     mass=1.479, width=0.080, spin=0, resonance_radius=R, parent_radius=R),
    ], normalization_resolution=350)

model_plus = build_model(channel_plus, +1)
model_minus = build_model(channel_minus, -1)
truth = {p.name: float(p.value) for p in model_plus.parameters}
print('free parameters:', sum(not p.fixed for p in model_plus.parameters))

## 3. Generate a joint charge + Dalitz toy

The total number of events is fixed, but the charge split is generated from
\[P(+)=I_+/(I_++I_-),\qquad P(-)=I_-/(I_++I_-).\]

In [ ]:
N_POOL, N_TOTAL = 500_000, 120_000
pool_plus = model_plus.generate_phase_space(N_POOL, seed=78012004)
pool_minus = model_minus.generate_phase_space(N_POOL, seed=78012005)

w_plus = pool_plus.weights * model_plus.intensity(pool_plus.as_dict(), truth)
w_minus = pool_minus.weights * model_minus.intensity(pool_minus.as_dict(), truth)

rate_cache_plus = model_plus.prepare_cache(pool_plus)
rate_cache_minus = model_minus.prepare_cache(pool_minus)
I_plus = float(rate_cache_plus.normalization(truth))
I_minus = float(rate_cache_minus.normalization(truth))
p_plus = I_plus/(I_plus+I_minus)

rng = np.random.default_rng(78012006)
N_PLUS = rng.binomial(N_TOTAL, p_plus)
N_MINUS = N_TOTAL - N_PLUS
print(f'I+={I_plus:.6f}, I-={I_minus:.6f}, P(+)={p_plus:.5f}')
print(f'N+={N_PLUS}, N-={N_MINUS}, raw asym={(N_MINUS-N_PLUS)/N_TOTAL:+.5f}')

toy_plus = weighted_resample(jax.random.key(78012007), pool_plus, w_plus, N_PLUS, replace=True)
toy_minus = weighted_resample(jax.random.key(78012008), pool_minus, w_minus, N_MINUS, replace=True)

## 4. Dalitz plots and local CP-asymmetry map

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, toy, title in [(axes[0],toy_plus,r'$B^+$'),(axes[1],toy_minus,r'$B^-$')]:
    h=ax.hist2d(np.asarray(toy.s13),np.asarray(toy.s23),bins=100)
    fig.colorbar(h[3],ax=ax,label='events')
    ax.set(xlabel=r'$s_{13}=m^2(K^\pm\pi^\mp)$ [GeV$^2$]', ylabel=r'$s_{23}=m^2(\pi^+\pi^-)$ [GeV$^2$]', title=title)
plt.show()

xbins=np.linspace(min(np.min(toy_plus.s13),np.min(toy_minus.s13)),max(np.max(toy_plus.s13),np.max(toy_minus.s13)),65)
ybins=np.linspace(min(np.min(toy_plus.s23),np.min(toy_minus.s23)),max(np.max(toy_plus.s23),np.max(toy_minus.s23)),65)
Hp,_,_=np.histogram2d(np.asarray(toy_plus.s13),np.asarray(toy_plus.s23),bins=(xbins,ybins))
Hm,_,_=np.histogram2d(np.asarray(toy_minus.s13),np.asarray(toy_minus.s23),bins=(xbins,ybins))
A=(Hm-Hp)/np.maximum(Hm+Hp,1)
fig,ax=plt.subplots(figsize=(7,5.5),constrained_layout=True)
im=ax.pcolormesh(xbins,ybins,A.T,vmin=-1,vmax=1)
fig.colorbar(im,ax=ax,label=r'$(N_- - N_+)/(N_- + N_+)$')
ax.set(xlabel=r'$s_{13}$ [GeV$^2$]',ylabel=r'$s_{23}$ [GeV$^2$]',title='Raw local CP asymmetry')
plt.show()

## 5. $s_{13}$ and $s_{23}$ projections with raw asymmetries

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12,8),constrained_layout=True)
for col,(attr,label) in enumerate([('s13',r'$s_{13}=m^2(K\pi)$'),('s23',r'$s_{23}=m^2(\pi\pi)$')]):
    vp=np.asarray(getattr(toy_plus,attr)); vm=np.asarray(getattr(toy_minus,attr))
    bins=np.linspace(min(vp.min(),vm.min()),max(vp.max(),vm.max()),100)
    hp,e=np.histogram(vp,bins=bins); hm,_=np.histogram(vm,bins=bins); x=0.5*(e[:-1]+e[1:])
    axes[0,col].step(x,hp,where='mid',label=r'$B^+$'); axes[0,col].step(x,hm,where='mid',label=r'$B^-$'); axes[0,col].legend(); axes[0,col].set_ylabel('events')
    axes[1,col].axhline(0,lw=.8); axes[1,col].step(x,(hm-hp)/np.maximum(hm+hp,1),where='mid'); axes[1,col].set(xlabel=label+' [GeV$^2$]',ylabel=r'$(N_- - N_+)/(N_- + N_+)$')
plt.show()

## 6. Joint CP likelihood and one-start closure fit

Each charge is coherent internally, but both charges share the same global normalization $I_+ + I_-$ through `CPJointNLL`.

In [ ]:
cache_plus=model_plus.prepare_cache(toy_plus)
cache_minus=model_minus.prepare_cache(toy_minus)
objective=CPJointNLL(cache_plus,cache_minus)
parameters=model_plus.parameters
fitter=Minimizer(objective,parameters,tolerance=1e-5,verbose=1)
start=fitter.random_start(seed=20260830)
result=fitter.fit(start_values=start,simplex=False,ncall=50000)
print(result.fmin)
print('charge probabilities truth:', tuple(float(v) for v in objective.charge_probabilities(truth)))

## 7. Pulls of all floating Cartesian parameters

In [ ]:
free=[p for p in parameters if not p.fixed]
fit={p.name:float(result.values[p.name]) for p in free}; err={p.name:float(result.errors[p.name]) for p in free}
fit_all={**truth,**fit}
pulls=np.array([(fit[p.name]-truth[p.name])/err[p.name] for p in free])

print(f"{'parameter':18s} {'truth':>9s} {'start':>9s} {'fit':>9s} {'err':>9s} {'pull':>8s}")
for p,pull in zip(free,pulls):
    print(f"{p.name:18s} {truth[p.name]:9.4f} {start[p.name]:9.4f} {fit[p.name]:9.4f} {err[p.name]:9.4f} {pull:8.2f}")

fig,ax=plt.subplots(figsize=(13,5),constrained_layout=True)
ax.axhline(0,lw=.8); ax.axhline(1,ls='--',lw=.7); ax.axhline(-1,ls='--',lw=.7)
ax.scatter(np.arange(len(free)),pulls)
ax.set_xticks(np.arange(len(free)),[p.name for p in free],rotation=75,ha='right')
ax.set(ylabel='pull',title='Closure pulls: all floating CP parameters')
plt.show()

## 8. Component $A_{CP}$ and Argand closure

In [ ]:
def complex_pair(c,values):
    return complex(c.for_charge(+1).value(values)),complex(c.for_charge(-1).value(values))

def acp(c,values):
    cp,cm=complex_pair(c,values)
    return (abs(cm)**2-abs(cp)**2)/(abs(cm)**2+abs(cp)**2)

names=list(coeff)
acp_truth=np.array([acp(coeff[n],truth) for n in names])
acp_fit=np.array([acp(coeff[n],fit_all) for n in names])

fig,ax=plt.subplots(figsize=(11,5),constrained_layout=True)
x=np.arange(len(names)); ax.axhline(0,lw=.8)
ax.scatter(x,acp_truth,marker='x',s=80,label='truth'); ax.scatter(x,acp_fit,marker='o',s=45,label='fit')
ax.set_xticks(x,names,rotation=45,ha='right'); ax.set(ylabel=r'$A_{CP}$',title='Component CP asymmetries'); ax.legend()
plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,q,label in [(axes[0],+1,r'$B^+$'),(axes[1],-1,r'$B^-$')]:
    for name in names:
        zt=complex(coeff[name].for_charge(q).value(truth)); zf=complex(coeff[name].for_charge(q).value(fit_all))
        ax.plot([zt.real,zf.real],[zt.imag,zf.imag],'-o',ms=3)
        ax.text(zf.real,zf.imag,name,fontsize=7)
    ax.axhline(0,lw=.6); ax.axvline(0,lw=.6); ax.set(xlabel='Re(c)',ylabel='Im(c)',title=label); ax.set_aspect('equal',adjustable='datalim')
plt.show()

## 9. Truth vs fit global charge fraction

This is the diagnostic that would be absent in two independently normalized charge likelihoods.

In [ ]:
p_truth=np.array([float(v) for v in objective.charge_probabilities(truth)])
p_fit=np.array([float(v) for v in objective.charge_probabilities(fit_all)])
observed=np.array([N_PLUS,N_MINUS])/N_TOTAL
fig,ax=plt.subplots(figsize=(7,4),constrained_layout=True)
x=np.arange(2); w=.25
ax.bar(x-w,observed,w,label='toy observed'); ax.bar(x,p_truth,w,label='truth PDF'); ax.bar(x+w,p_fit,w,label='fit PDF')
ax.set_xticks(x,[r'$B^+$',r'$B^-$']); ax.set(ylabel='charge fraction',title='Joint-normalization charge fractions'); ax.legend()
plt.show()